---
## TUGAS MANDIRI 

### Konteks / Skenario

Manajemen "Kopi Nusantara" ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [18]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Instruksi Pengerjaan
---
Pembuatan `SparkSession` dan membaca `tugas11_penjualan.csv`

In [19]:
# Persiapan ulang untuk latihan
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


spark = SparkSession.builder.appName("Tugas11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_penjualan = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print("Siap untuk latihan. Total baris:", df_penjualan.count())
print()
df_penjualan.printSchema()

Siap untuk latihan. Total baris: 2500

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

In [25]:
# StringIndexer mengubah nilai teks menjadi indeks numerik
indexer_menu   = StringIndexer(inputCol="nama_menu", outputCol="menu_idx").fit(df_penjualan)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_penjualan)
indexer_waktu  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df_penjualan)

# fit() dulu (mempelajari daftar kategori), baru transform() (menerapkannya)
df_indexed = indexer_menu.transform(df_penjualan)
df_indexed = indexer_cabang.transform(df_indexed)
df_indexed = indexer_waktu.transform(df_indexed)

# Menggabungkan seluruh kolom numerik (termasuk hasil StringIndexer) menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



---
**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

In [27]:
# randomSplit membagi data secara acak — 75% untuk latihan (training), 25% untuk pengujian (testing)
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")
print()

# Mendefinisikan model: featuresCol menunjuk ke kolom fitur, labelCol menunjuk ke kolom target yang diprediksi
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

# fit() melatih model menggunakan data training 
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661

Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


---
**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?


In [30]:
# Membuat prediksi
predictions = model.transform(test)

# RMSE (Root Mean Squared Error) -- rata-rata besar kesalahan prediksi, dalam satuan yang sama dengan target (Rupiah)
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# R-squared -- seberapa besar proporsi variasi data yang berhasil "dijelaskan" oleh model (0 = buruk, 1 = sempurna)
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)


**Kalimat interpretasi :** Karena `total_penjualan` dihitung dari `qty` x `harga_satuan`, kesalahan sekitar 10 ribu per pesanan masih tergolong wajar. R² di atas 0.8 (sebesar 0.91) menunjukkan model cukup baik menangkap pola hubungan antara fitur-fitur dan pendapatan, sehingga layak dipakai untuk memprediksi omzet.

---
**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [32]:
# Model kedua dengan dua fitur
assembler_2 = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)
df_final = assembler_2.transform(df_indexed)

# Melatih model
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
lr = LinearRegression(featuresCol="features_2", labelCol="total_penjualan") 
model_2 = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model_2.coefficients)
print("Intercept:", model_2.intercept)
print()

# Membuat prediksi
predictions = model_2.transform(test)

# Evaluasi model regresi
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

Model berhasil dilatih!
Koefisien tiap fitur: [22048.50478817013,3.021049809988266]
Intercept: -66397.03118601872

RMSE : Rp 10,896
R²   : 0.9108  (91.1% variasi data berhasil dijelaskan model)


**Kesimpulan :** Pada model pertama dengan 5 fitur, memiliki R² sebesar 0.9107. Hal ini tidak benar-benar meningkatkan akurasi prediksi secara berarti karena pada model kedua hanya dengan 2 fitur memiliki R² sebesar 0.9108 dengan selisih 0.0001 dengan model yang pertama.

---
**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [35]:
# Menambahkan skenario pesanan
skenario = spark.createDataFrame([
    ("Americano", "Magelang", "Malam", 3, 15000),
    ("Roti Bakar", "Semarang", "Siang", 4, 30000),
    ("Kentang Goreng", "Yogyakarta", "Pagi", 2, 20000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training
skenario = indexer_menu.transform(skenario)
skenario = indexer_cabang.transform(skenario)
skenario = indexer_waktu.transform(skenario)
skenario = assembler.transform(skenario)

final = model.transform(skenario)
final.select("qty", "harga_satuan", "nama_menu", "cabang", "waktu_pesan", "prediction").show(truncate=False)

+---+------------+--------------+----------+-----------+-----------------+
|qty|harga_satuan|nama_menu     |cabang    |waktu_pesan|prediction       |
+---+------------+--------------+----------+-----------+-----------------+
|3  |15000       |Americano     |Magelang  |Malam      |44920.26396595164|
|4  |30000       |Roti Bakar    |Semarang  |Siang      |112716.0910799979|
|2  |20000       |Kentang Goreng|Yogyakarta|Pagi       |37311.77948508995|
+---+------------+--------------+----------+-----------+-----------------+



---

## Menutup SparkSession

In [36]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
